In [ ]:
import pandas as pd
import numpy as np

# Loading weather data
df = pd.read_parquet("data/raw/aws_clean_baseline.parquet")

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df_eval=df.copy()
df_eval["is_anomoly"]=0
df_eval["anomaly_type"] = "normal"
df_eval["affected_sensor"] = "none"
df_eval.head()

In [ ]:
#adding spikes 
np.random.seed(42)
n_spikes=int(len(df_eval)*0.004)#so adding 0.4 % spikes(roughly) right now
spike_indices = np.random.choice(df_eval.index, size=n_spikes, replace=False)#selecting which indexes to add spikes at
for index in spike_indices:
    sensor=np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])
    if sensor=="temp_c":
        spiked_difference = np.random.choice([+30.0, +40.0, -25.0])
        df_eval.loc[index, "temp_c"] += spiked_difference
    elif sensor=="humidity_pct":
        df_eval.loc[index,"humidity_pct"]=np.random.choice([130.0,145.0,-10.0])#setting humidity to an impossible value 
    elif sensor =="pressure_hpa":
        spiked_difference=np.random.choice([+80.0, -80.0])
        df_eval.loc[index,"pressure_hpa"]+=spiked_difference
    df_eval.loc[index,"is_anomoly"]=1
    df_eval.loc[index,"anomoly_type"]="spike"
    df_eval.loc[index, "affected_sensor"] = sensor

df_eval["is_anomoly"].value_counts()#is giving 1052 rows which is roghly 0.4% so evrythings fine
df_eval[df_eval["anomoly_type"]=="spike"][["temp_c","pressure_hpa","humidity_pct","affected_sensor"]]



In [ ]:
# Injecting Frozen (Stuck) Sensors
np.random.seed(42) 
n_frozen_events = 20 
duration_hours = 12 #frozen for how many hours

stations = df_eval["station_id"].unique()

for i in range(n_frozen_events):
    
    station = np.random.choice(stations)#station chunega
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()#getting the station data, index here is basically hoir by hour data

    if len(station_indices) > duration_hours:
        
        start_pos = np.random.randint(0, len(station_indices) - duration_hours)#so that actual me duration hours tak ka data available ho
        event_idx = station_indices[start_pos : start_pos + duration_hours]
        
        
        sensor = np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])#selecting the sensor to freeze
     #currently assuming ki value remains feezed exactly , that is no single variation throighout the timeline but small variations ke liye well add secodn derivative later on
        frozen_val = df_eval.loc[event_idx[0], sensor]#overriting the first hour value at every next hour in diuration
        df_eval.loc[event_idx, sensor] = frozen_val
    
        df_eval.loc[event_idx, "is_anomoly"] = 1
        df_eval.loc[event_idx, "anomoly_type"] = "frozen_sensor"
        df_eval.loc[event_idx, "affected_sensor"] = sensor
df_eval["anomoly_type"].value_counts()

In [ ]:
#now adding the second derivative based anomolies
#basically, assume we have an insulation issue, that is there is something which is blocking sensor's contact wiht the external environment, be it a random object stuck on it like a clot, or maybe shadow due to a huge object kept in front of it ir maybe dust/uce accumulation
# in all such cases, the effect of external factors on changing the poarameters will reduce drastically, so say temp was to be increased due to sun but cloth got stuck, it wiull still increase but as cloth will boco most of rays, the incerase will be really small, so the graph will become largely linerly increasing (because the factor that was causing the change got supressed so the changes are negligible ) and hence the second derivative will be very clkose to zero
